In [42]:
import geopandas as gpd
import pandas as pd
import numpy as np
from libpysal.weights import Queen, KNN
import matplotlib.pyplot as plt
from shapely import STRtree
from pyprojroot import here

In [43]:
INTERIM_DATA_DIR = here() / "data" / "interim"
RESULTS_DATA_DIR = here() / "results" 
RAW_DATA_DIR = here() / "data" / "raw"

In [44]:
gdf = gpd.read_file(RAW_DATA_DIR/"geographic/municipalities/00mun.shp")

print(f"Municipalities : {len(gdf)}")
print(f"CRS            : {gdf.crs.name}")
print(gdf[["CVEGEO", "CVE_ENT", "NOMGEO"]].head(5))

Municipalities : 2478
CRS            : MEXICO_ITRF_2008_LCC
  CVEGEO CVE_ENT                     NOMGEO
0  01001      01             Aguascalientes
1  01008      01         San José de Gracia
2  01011      01  San Francisco de los Romo
3  01009      01                   Tepezalá
4  01007      01            Rincón de Romos


### Queen based contiguity

In [45]:
w_queen = Queen.from_dataframe(gdf, silence_warnings=True)
print(f"Queen adjacencies  : {sum(len(v) for v in w_queen.neighbors.values()) // 2}")
print(f"Avg neighbors      : {w_queen.mean_neighbors:.2f}")
print(f"Islands (0 nbrs)   : {sum(1 for v in w_queen.neighbors.values() if len(v) == 0)}")

C:\Users\franc\AppData\Local\Temp\ipykernel_79012\577925112.py:1: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w_queen = Queen.from_dataframe(gdf, silence_warnings=True)


Queen adjacencies  : 6278
Avg neighbors      : 5.07
Islands (0 nbrs)   : 52


In [46]:
labels = gdf["CVEGEO"].tolist()        # use CVEGEO as the key
n = len(labels)

adj_queen = pd.DataFrame(
    np.zeros((n, n), dtype=np.int8),
    index=labels, columns=labels
)
for i, nbrs in w_queen.neighbors.items():
    for j in nbrs:
        adj_queen.iloc[i, j] = 1

assert (adj_queen == adj_queen.T).all().all(), "Queen matrix not symmetric!"
print(f"Queen matrix shape : {adj_queen.shape}")

Queen matrix shape : (2478, 2478)


In [47]:
adj_true = adj_queen.copy()

for a, b in true_touch_pairs:
    cvegeo_a = gdf["CVEGEO"].iloc[a]
    cvegeo_b = gdf["CVEGEO"].iloc[b]
    adj_true.loc[cvegeo_a, cvegeo_b] = 1
    adj_true.loc[cvegeo_b, cvegeo_a] = 1

assert (adj_true == adj_true.T).all().all(), "True-touch matrix not symmetric!"

added = int((adj_true.values.sum() - adj_queen.values.sum()) / 2)
print(f"Adjacencies added by true-touch patch : {added}")
print(f"Total adjacencies — Queen only         : {int(adj_queen.values.sum() // 2)}")
print(f"Total adjacencies — Queen + true-touch : {int(adj_true.values.sum() // 2)}")

Adjacencies added by true-touch patch : 974
Total adjacencies — Queen only         : 6278
Total adjacencies — Queen + true-touch : 7252


In [48]:
queen_neighbor_count = adj_queen.sum(axis=1)
islands_queen    = queen_neighbor_count[queen_neighbor_count == 0].index.tolist()
true_neighbor_count = adj_true.sum(axis=1)
islands_true = true_neighbor_count[true_neighbor_count == 0].index.tolist()

print(f"Islands in Queen only          : {len(islands_queen)}")
print(f"Islands after true-touch patch : {len(islands_true)}")

if islands_true:
    print("\n=== Still isolated after true-touch patch (candidates for option 3) ===")
    print(meta.loc[islands_true])
else:
    print("\n✓ No islands remain — connectivity achieved via true-touch patch alone.")

Islands in Queen only          : 52
Islands after true-touch patch : 0

✓ No islands remain — connectivity achieved via true-touch patch alone.


### Sanity check, sample any municipality

In [49]:
def show_neighbors(cvegeo, adj=adj_true, gdf=gdf):
    if cvegeo not in adj.index:
        print(f"{cvegeo} not found in matrix.")
        return
    nbrs = adj.loc[cvegeo][adj.loc[cvegeo] == 1].index.tolist()
    meta_lookup = gdf.set_index("CVEGEO")[["CVE_ENT", "NOMGEO"]]
    own_name = meta_lookup.loc[cvegeo, "NOMGEO"]
    print(f"{cvegeo} — {own_name}  ({len(nbrs)} neighbors)")
    for c in nbrs:
        print(f"  {c}  {meta_lookup.loc[c, 'NOMGEO']}")

show_neighbors("06009")

06009 — Tecomán  (5 neighbors)
  06002  Colima
  06006  Ixtlahuacán
  06001  Armería
  06004  Coquimatlán
  16014  Coahuayana


In [50]:
adj_combined = adj_true.copy()  

adj_combined.to_csv(INTERIM_DATA_DIR/"adj_combined.csv")
print("Saved → adj_combined.csv")
print(f"Shape       : {adj_combined.shape}")
print(f"Adjacencies : {int(adj_combined.values.sum() // 2)}")
print(f"Islands     : {int((adj_combined.sum(axis=1) == 0).sum())}")

Saved → adj_combined.csv
Shape       : (2478, 2478)
Adjacencies : 7252
Islands     : 0
